# 04 — Cribado por título y resumen

Cribado inicial del mapa mundial de evidencia sobre cambio climático y pesquerías marinas, con prioridad analítica para pequeños pelágicos y anchoveta peruana.

La decisión usa únicamente título, resumen y metadatos:

- `include`: los tres criterios son `yes`;
- `exclude`: al menos uno es claramente `no` y se consigna una razón;
- `uncertain`: al menos uno es `unclear`; la fuente pasa a aclaración o texto completo.

El triaje por palabras clave solo prioriza el orden de revisión; no decide elegibilidad.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evidence_review.screening import (
    apply_keyword_triage,
    build_screening_corpus,
    completed_binary_decisions,
    export_prompt_jsonl,
    initialise_screening_sheet,
    load_screening_config,
    screening_summary,
    select_pilot_sample,
    validate_screening_sheet,
)

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)
print(f"Project root: {ROOT}")


## 1. Configuración y corpus deduplicado


In [ ]:
config = load_screening_config(ROOT / "config" / "screening.yml")
retained_path = INTERIM / "deduplicated_sources.csv"
if not retained_path.exists():
    raise FileNotFoundError(
        "Falta data/interim/deduplicated_sources.csv. Ejecuta primero el notebook 03."
    )
retained = pd.read_csv(retained_path).fillna("")
if retained.empty:
    raise ValueError("El corpus deduplicado está vacío.")
print(f"Retained sources: {len(retained)}")
print("Decisions:", ", ".join(config["decisions"]))
print("Criterion values:", ", ".join(config["criterion_values"]))


## 2. Preparar título, resumen y triaje diagnóstico


In [ ]:
screening_corpus = apply_keyword_triage(
    build_screening_corpus(retained),
    config,
)
corpus_path = INTERIM / "title_abstract_screening_corpus.csv"
screening_corpus.to_csv(corpus_path, index=False, encoding="utf-8-sig")

print(f"Sources prepared: {len(screening_corpus)}")
print(f"Abstract available: {int(screening_corpus['abstract_available'].sum())}")
print(f"Abstract missing/short: {int((~screening_corpus['abstract_available']).sum())}")
display(
    screening_corpus["triage_priority"]
    .value_counts(dropna=False)
    .rename_axis("triage_priority")
    .reset_index(name="sources")
)
display(screening_corpus[[
    "source_id", "title", "year", "abstract_available", "triage_priority",
    "climate_keyword_hits", "fisheries_keyword_hits",
    "priority_taxa_keyword_hits", "suggested_priority_groups",
]].head(15))


## 3. Crear o actualizar la hoja de trabajo

Las decisiones previas se conservan mediante `source_id`. Las filas nuevas quedan sin decisión.


In [ ]:
working_path = INTERIM / "title_abstract_screening_working.csv"
existing = (
    pd.read_csv(working_path).fillna("")
    if working_path.exists()
    else pd.DataFrame()
)
screening_sheet = initialise_screening_sheet(screening_corpus, existing)
screening_sheet.to_csv(working_path, index=False, encoding="utf-8-sig")
print(f"Working rows: {len(screening_sheet)}")
print(f"Saved: {working_path.relative_to(ROOT)}")
display(screening_summary(screening_sheet))


## 4. Libro de códigos

Para cada fuente completa:

- `fisheries_or_marine_relevant`: `yes`, `no` o `unclear`;
- `climate_environment_or_adaptation_element`: `yes`, `no` o `unclear`;
- `contributes_codable_evidence`: `yes`, `no` o `unclear`;
- `decision`: `include`, `exclude` o `uncertain`;
- `exclusion_reason`: solo cuando `decision = exclude`;
- `priority_groups`: uno o varios códigos separados por `|`.

La ausencia de pequeños pelágicos no excluye automáticamente una fuente transferible de otra pesquería marina.


In [ ]:
criteria = pd.DataFrame([
    {"criterion": name, "question": value["question"]}
    for name, value in config["eligibility_criteria"].items()
])
display(criteria)
display(pd.DataFrame({"allowed_exclusion_reason": config["exclusion_reasons"]}))
display(pd.DataFrame({"allowed_priority_group": config["priority_groups"]}))


## 5. Muestra piloto

Se revisan todas las fuentes cuando el corpus tiene hasta 50 registros. Para corpus mayores se seleccionan 30 de forma reproducible y estratificada por prioridad diagnóstica.


In [ ]:
pilot_n = len(screening_sheet) if len(screening_sheet) <= 50 else 30
pilot_sample = select_pilot_sample(screening_sheet, n=pilot_n, random_state=42)
pilot_path = INTERIM / "title_abstract_screening_pilot.csv"
pilot_sample.to_csv(pilot_path, index=False, encoding="utf-8-sig")
print(f"Pilot sources: {len(pilot_sample)}")
print(f"Saved: {pilot_path.relative_to(ROOT)}")
display(pilot_sample[[
    "source_id", "title", "abstract", "triage_priority",
    "suggested_priority_groups", "decision",
]])


## 6. Prompts auditables opcionales

Esta celda no llama a ninguna API. Solo genera un JSONL con un prompt por fuente. Cualquier decisión automática requiere validación humana.


In [ ]:
prompt_path = INTERIM / "title_abstract_screening_prompts.jsonl"
export_prompt_jsonl(pilot_sample, config, prompt_path)
print(f"Prompt records: {len(pilot_sample)}")
print(f"Saved: {prompt_path.relative_to(ROOT)}")


## 7. Validar la hoja editada

Edita `data/interim/title_abstract_screening_working.csv`, guarda el archivo y vuelve a ejecutar desde esta celda. Las filas todavía no revisadas pueden permanecer vacías.


In [ ]:
reviewed = pd.read_csv(working_path).fillna("")
issues = validate_screening_sheet(reviewed, config)
issues_path = INTERIM / "title_abstract_screening_issues.csv"
issues.to_csv(issues_path, index=False, encoding="utf-8-sig")
print(f"Validation issues: {len(issues)}")
display(issues.head(50))
display(screening_summary(reviewed))


## 8. Exportar decisiones válidas


In [ ]:
if issues.empty:
    valid_rows = reviewed
else:
    invalid_ids = set(issues["source_id"].dropna().astype(str))
    valid_rows = reviewed.loc[~reviewed["source_id"].astype(str).isin(invalid_ids)]

binary = completed_binary_decisions(valid_rows)
binary.to_csv(
    INTERIM / "screening_decisions_working.csv",
    index=False,
    encoding="utf-8-sig",
)
for decision in ("include", "uncertain", "exclude"):
    reviewed.loc[reviewed["decision"] == decision].to_csv(
        INTERIM / f"title_abstract_{decision}d.csv",
        index=False,
        encoding="utf-8-sig",
    )
print(f"Binary decisions exported: {len(binary)}")
print(f"Included: {(reviewed['decision'] == 'include').sum()}")
print(f"Uncertain: {(reviewed['decision'] == 'uncertain').sum()}")
print(f"Excluded: {(reviewed['decision'] == 'exclude').sum()}")


## Criterio para avanzar

La fase de texto completo comienza cuando el piloto esté completado y sin errores lógicos, cada exclusión tenga una razón válida, las fuentes `uncertain` estén identificadas y la búsqueda muestre sensibilidad suficiente para anchoveta, otros pequeños pelágicos y evidencia transferible.
